In [1]:
import pandas as pd

orders = pd.read_csv('data/orders_clean.csv')

orders.shape

(800, 10)

In [2]:
orders.dtypes

order_id           str
customer_id        str
order_date         str
product            str
category           str
unit_price     float64
quantity         int64
express           bool
coupon_code        str
total          float64
dtype: object

In [3]:
orders = pd.read_csv('data/orders_clean.csv', parse_dates=['order_date'])

print(orders['order_date'].dtype)

datetime64[us]


## Missing values in the orders

In [4]:
orders.isna().sum()

order_id         0
customer_id      0
order_date       4
product          0
category         0
unit_price       0
quantity         0
express          0
coupon_code    635
total            0
dtype: int64

In [5]:
(orders.isna().mean() * 100).round(1)

order_id        0.0
customer_id     0.0
order_date      0.5
product         0.0
category        0.0
unit_price      0.0
quantity        0.0
express         0.0
coupon_code    79.4
total           0.0
dtype: float64

In [6]:
orders[orders['order_date'].isna()]

,order_id,customer_id,order_date,product,category,unit_price,quantity,express,coupon_code,total
47,ORD-20048,C0082,NaT,T-shirt,Clothing,199.0,1,True,NaN,199.0
95,ORD-20096,C0107,NaT,Yoga mat,Sports,249.0,1,False,FRIEND15,249.0
522,ORD-20523,C0075,NaT,Yoga mat,Sports,249.0,2,True,NaN,498.0
763,ORD-20764,C0030,NaT,Dumbbell set,Sports,599.0,3,False,SUMMER20,1797.0


**Missing for a reason**

**Missing by accident**

In [7]:
orders['used_coupon'] = orders['coupon_code'].notna()  # True where there IS a value

orders['used_coupon'].value_counts()

used_coupon
False    635
True     165
Name: count, dtype: int64

## A new table: the customers

In [8]:
customers_raw = pd.read_csv('data/customers.csv')
customers = customers_raw.copy()

customers.shape

(297, 3)

In [9]:
customers.head(8)

,customer_id,city,age
0,C0001,Göteborg,41.0
1,C0002,Västerås,999.0
2,C0003,Stockholm,38.0
3,C0004,Västerås,54.0
4,C0005,Göteborg,31.0
5,C0006,NaN,40.0
6,C0007,Stockholm,39.0
7,C0008,Stockholm,50.0


In [10]:
customers.info()

<class 'pandas.DataFrame'>
RangeIndex: 297 entries, 0 to 296
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   customer_id  297 non-null    str    
 1   city         277 non-null    str    
 2   age          267 non-null    float64
dtypes: float64(1), str(2)
memory usage: 7.1 KB


In [11]:
customers['age'].describe()

count    267.000000
mean      69.947566
std      174.235314
min        0.000000
25%       30.000000
50%       38.000000
75%       46.500000
max      999.000000
Name: age, dtype: float64

In [12]:
print((customers['age'] == 999).sum())
print((customers['age'] == 0).sum())


9
3


In [13]:
print(customers.duplicated().sum())
print(customers['customer_id'].duplicated().sum())


5
5


1. **20 cities are missing**
2. **30 ages are missing**
3. **9 ages are 999, a placeholder**
4. **3 ages are 0, which is impossible**
5. **5 duplicated rows**


## What do we do with the missing values?

3 choices:

- **Remove**
- **Fill in**
- **Leave them**



In [14]:
customers['age'] = customers['age'].where(customers['age'].between(15, 100))

print(customers['age'].isna().sum())

42


In [15]:
customers['age'].describe()


count    255.000000
mean      37.980392
std       11.172397
min       16.000000
25%       30.000000
50%       38.000000
75%       45.000000
max       70.000000
Name: age, dtype: float64

### Removing rows: dropna()

In [16]:
print(customers.dropna().shape)# Any column
print(customers.dropna(subset=['city']).shape) # City is missing
print(customers.dropna(subset=['age']).shape) # Age is missing


(235, 3)
(277, 3)
(255, 3)


### Filling in: fillna()

In [17]:
customers['city'] = customers['city'].fillna('Unknown')

customers['city'].value_counts()

city
Stockholm    88
Göteborg     50
Malmö        35
Uppsala      33
Linköping    21
Unknown      20
Västerås     19
Örebro       19
Umeå         12
Name: count, dtype: int64

class 1 -> avg = 38   between 36 and 40
class 2 -> avg = 38   between 20 and 60

how spread out the ages are

standard deviation: std()

small number = similar values,

big numbers = spread out values



In [18]:
age_median = customers['age'].median()
age_filled = customers['age'].fillna(age_median)

print(age_median)
print(customers['age'].mean().round(2), age_filled.mean().round(2))
print(customers['age'].std().round(2), age_filled.std().round(2))


38.0
37.98 37.98
11.17 10.35


Summarise 3 choices:

- **Remove** (dropna): simple, but you lose the rest of the row. Use subset to be careful.
- **Fill inn** (fillna): good when there is a natural value, like `'Unknown'`. Be careful with numbers: the filled-in values is a guess, and it changes the spread.
- **Leve it**: often the best for numbers. Pandas skips missing values in calculations.

## Merge: putting two tables together

In [21]:
orders_small = pd.DataFrame({
    'order_id': ['A1', 'A2', 'A3'],
    'customer_id': ['C1', 'C2', 'C9']
})

customers_small = pd.DataFrame({
    'customer_id': ['C1', 'C2', 'C3'],
    'city': ['Stockholm', 'Malmö', 'Uppsala']
})


orders_small.merge(customers_small, on='customer_id', how='left')

,order_id,customer_id,city
0,A1,C1,Stockholm
1,A2,C2,Malmö
2,A3,C9,NaN


In [22]:
orders_small.merge(customers_small, on='customer_id', how='inner')


,order_id,customer_id,city
0,A1,C1,Stockholm
1,A2,C2,Malmö


In [23]:
print(orders.shape)
print(customers.shape)

(800, 11)
(297, 3)


In [24]:
merged = orders.merge(customers, on='customer_id', how='left')

In [25]:
merged.shape

(813, 13)

## The trap: duplicates in the key

In [26]:
customers[customers['customer_id'].duplicated(keep=False)].sort_values('customer_id')

,customer_id,city,age
8,C0009,Linköping,53.0
9,C0009,Linköping,53.0
10,C0010,Uppsala,47.0
11,C0010,Uppsala,47.0
69,C0070,Stockholm,38.0
70,C0070,Stockholm,38.0
97,C0097,Linköping,35.0
98,C0097,Linköping,35.0
197,C0199,Malmö,31.0
198,C0199,Malmö,31.0


In [27]:
print(merged['order_id'].duplicated().sum())    # orders that appear morre than once
print(merged['total'].sum().round(1))           # total revenue after the merge
print(orders['total'].sum().round(1))           # total revenue before the merge
print((merged['total'].sum() - orders['total'].sum()).round(1))     #the difference

13
585457.0
572783.8
12673.2


In [28]:
orders.merge(customers, on='customer_id', how='left', validate='many_to_one')

MergeError: Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
 customer_id
      C0009
      C0010
      C0070
      C0097
      C0199 ...

In [29]:
customers = customers.drop_duplicates().reset_index(drop=True)

print(customers.shape)
print(customers['customer_id'].is_unique)

(292, 3)
True


In [ ]:
merged = orders.merge(customers, on='customer_id', how='left', validate='many_to_one') # Many orders for ONE customer. Each customer may appear only once

print(merged.shape)
print(merged['total'].sum().round(1))


(800, 13)
572783.8


## What did we lose? Using the merged table!

indicator=True

_merge

In [ ]:
#Find out which order did NOT find a customer

merged = orders.merge(customers, on='customer_id', how='left', validate='many_to_one', indicator=True) 

merged['_merge'].value_counts()

# both The order found it customer
# left_only The order has no customer in the customer table.
# right_only A customer without orders (only with how='right' or 'outer)


_merge
both          771
left_only      29
right_only      0
Name: count, dtype: int64

In [32]:
no_customer = merged[merged['_merge'] == 'left_only']

print(no_customer['customer_id'].nunique())
print(no_customer['total'].sum().round(1))

8
17438.6


In [33]:
inner = orders.merge(customers, on='customer_id', how='inner', validate='many_to_one')

print(inner.shape)
print(inner['total'].sum().round(1))


(771, 13)
555345.2


which city buys the most?

In [34]:
merged['city'] = merged['city'].fillna('No customer record')

revenue_per_city = merged.groupby('city')['total'].sum().round(0).sort_values(ascending=False)

revenue_per_city

city
Stockholm             175321.0
Göteborg               92686.0
Uppsala                73317.0
Malmö                  65224.0
Örebro                 40994.0
Unknown                29417.0
Västerås               28780.0
Linköping              25058.0
Umeå                   24548.0
No customer record     17439.0
Name: total, dtype: float64

What is the average age of the customers in each city?

.agg

In [35]:
customers.groupby('city')['age'].agg(['count', 'mean']).round(1)

,count,mean
city,,
Göteborg,43,37.3
Linköping,19,43.1
Malmö,24,38.0
Stockholm,76,37.2
Umeå,11,40.6
Unknown,20,35.2
Uppsala,29,41.6
Västerås,15,37.2
Örebro,13,31.2
